# Mini Project 9 - Part 3: Transformer Fine-Tuning
## DistilBERT for 3-Class Content Moderation

**Goal:** Fine-tune DistilBERT to classify tweets as hate speech, offensive, or neither.

**Challenge:** Handle class imbalance and beat the TF-IDF baseline!

In [1]:
# Install packages if needed
%pip install transformers torch scikit-learn pandas numpy matplotlib seaborn tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 3060


## 1. Load Data

In [3]:
# Load splits
train_df = pd.read_csv('data/train.csv')
val_df = pd.read_csv('data/val.csv')
test_df = pd.read_csv('data/test.csv')

print(f"Train: {len(train_df)} samples")
print(f"Val:   {len(val_df)} samples")
print(f"Test:  {len(test_df)} samples")

# Use basic cleaned text (transformers handle noise better than TF-IDF)
X_train = train_df['text_clean'].values
y_train = train_df['class'].values

X_val = val_df['text_clean'].values
y_val = val_df['class'].values

X_test = test_df['text_clean'].values
y_test = test_df['class'].values

Train: 15849 samples
Val:   3963 samples
Test:  4954 samples


## 2. Load Tokenizer and Model

In [4]:
# Model configuration
MODEL_NAME = 'distilbert-base-uncased'
NUM_LABELS = 3
MAX_LENGTH = 128  # Tweets are short, 128 tokens is plenty

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✅ Loaded tokenizer: {MODEL_NAME}")

# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)
model = model.to(device)
print(f"✅ Loaded model with {NUM_LABELS} output classes")
print(f"   Model parameters: {sum(p.numel() for p in model.parameters()):,}")

✅ Loaded tokenizer: distilbert-base-uncased


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Loaded model with 3 output classes
   Model parameters: 66,955,779


## 3. Create Dataset Class

In [5]:
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        # Tokenize
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [6]:
# Create datasets
train_dataset = TweetDataset(X_train, y_train, tokenizer, MAX_LENGTH)
val_dataset = TweetDataset(X_val, y_val, tokenizer, MAX_LENGTH)
test_dataset = TweetDataset(X_test, y_test, tokenizer, MAX_LENGTH)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset:   {len(val_dataset)} samples")
print(f"Test dataset:  {len(test_dataset)} samples")

# Test one sample
sample = train_dataset[0]
print(f"\nSample shape:")
print(f"  input_ids: {sample['input_ids'].shape}")
print(f"  attention_mask: {sample['attention_mask'].shape}")
print(f"  label: {sample['label'].item()}")

Train dataset: 15849 samples
Val dataset:   3963 samples
Test dataset:  4954 samples

Sample shape:
  input_ids: torch.Size([128])
  attention_mask: torch.Size([128])
  label: 1


## 4. Create DataLoaders

Batch size = 16 (standard for DistilBERT on free Colab GPU)

In [7]:
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

Train batches: 991
Val batches:   248
Test batches:  310


## 5. Handle Class Imbalance with Weighted Loss

Compute class weights to handle imbalance.

In [8]:
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

print("Class weights:")
for i, weight in enumerate(class_weights):
    label = {0: 'Hate speech', 1: 'Offensive', 2: 'Neither'}[i]
    print(f"  Class {i} ({label:15s}): {weight:.4f}")

# Loss function with class weights
criterion = nn.CrossEntropyLoss(weight=class_weights)

Class weights:
  Class 0 (Hate speech    ): 5.7864
  Class 1 (Offensive      ): 0.4304
  Class 2 (Neither        ): 1.9853


## 6. Training Configuration

In [9]:
# Hyperparameters
EPOCHS = 4
LEARNING_RATE = 3e-5  # Standard for transformer fine-tuning
WARMUP_STEPS = 100

# Optimizer (AdamW is standard for transformers)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# Learning rate scheduler with warmup
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps
)

print(f"Training configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Total training steps: {total_steps}")
print(f"  Warmup steps: {WARMUP_STEPS}")

Training configuration:
  Epochs: 4
  Batch size: 16
  Learning rate: 3e-05
  Total training steps: 3964
  Warmup steps: 100


## 7. Training Loop

In [10]:
def train_epoch(model, data_loader, criterion, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    predictions = []
    true_labels = []
    
    progress_bar = tqdm(data_loader, desc='Training')
    
    for batch in progress_bar:
        # Move to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        logits = outputs.logits
        loss = criterion(logits, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping (prevent exploding gradients)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step()
        
        # Track metrics
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        predictions.extend(preds)
        true_labels.extend(labels.cpu().numpy())
        
        # Update progress bar
        progress_bar.set_postfix({'loss': loss.item()})
    
    avg_loss = total_loss / len(data_loader)
    accuracy = accuracy_score(true_labels, predictions)
    
    return avg_loss, accuracy

In [11]:
def evaluate(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0
    predictions = []
    true_labels = []
    all_probs = []
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            
            logits = outputs.logits
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            preds = np.argmax(probs, axis=1)
            
            predictions.extend(preds)
            true_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs)
    
    avg_loss = total_loss / len(data_loader)
    accuracy = accuracy_score(true_labels, predictions)
    
    return avg_loss, accuracy, predictions, true_labels, np.array(all_probs)

## 8. Train the Model!

In [ ]:
# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

best_val_acc = 0

print("="*70)
print("TRAINING START")
print("="*70)

for epoch in range(EPOCHS):
    print(f"\n{'='*70}")
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print(f"{'='*70}")
    
    # Train
    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, scheduler, device
    )
    
    # Validate
    val_loss, val_acc, _, _, _ = evaluate(model, val_loader, criterion, device)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Print results
    print(f"\nResults:")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '../data/best_model.pt')
        print(f"  ✅ New best model saved! (Val Acc: {val_acc:.4f})")

print("\n" + "="*70)
print("TRAINING COMPLETE!")
print("="*70)
print(f"Best validation accuracy: {best_val_acc:.4f}")

TRAINING START

Epoch 1/4


## 9. Plot Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, EPOCHS + 1)

# Loss
axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'], 'r-o', label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, history['train_acc'], 'b-o', label='Train Acc', linewidth=2)
axes[1].plot(epochs_range, history['val_acc'], 'r-o', label='Val Acc', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Training curves saved to ../data/training_curves.png")

## 10. Evaluate on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load('../data/best_model.pt'))
print("✅ Loaded best model")

# Evaluate on test set
test_loss, test_acc, test_preds, test_labels, test_probs = evaluate(
    model, test_loader, criterion, device
)

# Calculate F1 scores
test_f1_macro = f1_score(test_labels, test_preds, average='macro')
test_f1_weighted = f1_score(test_labels, test_preds, average='weighted')

print("\n" + "="*70)
print("DISTILBERT - FINAL TEST RESULTS")
print("="*70)
print(f"Test Loss:       {test_loss:.4f}")
print(f"Test Accuracy:   {test_acc:.4f}")
print(f"F1 (macro):      {test_f1_macro:.4f}")
print(f"F1 (weighted):   {test_f1_weighted:.4f}")
print("="*70)

## 11. Detailed Classification Report

In [ ]:
class_names = ['Hate speech', 'Offensive', 'Neither']
print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=class_names, digits=4))

## 12. Confusion Matrix

In [ ]:
cm = confusion_matrix(test_labels, test_preds)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_ylabel('True Label', fontsize=12)
axes[0].set_xlabel('Predicted Label', fontsize=12)
axes[0].set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')

# Percentages
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_ylabel('True Label', fontsize=12)
axes[1].set_xlabel('Predicted Label', fontsize=12)
axes[1].set_title('Confusion Matrix (Percentages)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/confusion_matrix_transformer.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Compare with Baseline

In [ ]:
# Load baseline results
import json
with open('../data/baseline_results.json', 'r') as f:
    baseline_results = json.load(f)

# Comparison table
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'F1 (macro)', 'F1 (weighted)'],
    'TF-IDF Baseline': [
        baseline_results['test_accuracy'],
        baseline_results['test_f1_macro'],
        baseline_results['test_f1_weighted']
    ],
    'DistilBERT': [
        test_acc,
        test_f1_macro,
        test_f1_weighted
    ]
})

comparison['Improvement'] = comparison['DistilBERT'] - comparison['TF-IDF Baseline']
comparison['% Improvement'] = (comparison['Improvement'] / comparison['TF-IDF Baseline'] * 100).round(2)

print("\n" + "="*80)
print("MODEL COMPARISON")
print("="*80)
print(comparison.to_string(index=False))
print("="*80)

## 14. Error Analysis: Misclassified Examples

In [ ]:
# Create results dataframe
results_df = test_df.copy()
results_df['predicted'] = test_preds
results_df['confidence'] = test_probs.max(axis=1)
results_df['correct'] = (results_df['class'] == results_df['predicted'])

# Get misclassified examples
errors = results_df[~results_df['correct']]

print(f"Total errors: {len(errors)} / {len(results_df)} ({len(errors)/len(results_df)*100:.2f}%)")
print("\nError breakdown by true class:")
for cls in [0, 1, 2]:
    label = {0: 'Hate speech', 1: 'Offensive', 2: 'Neither'}[cls]
    cls_errors = len(errors[errors['class'] == cls])
    cls_total = len(results_df[results_df['class'] == cls])
    print(f"  {label:15s}: {cls_errors:3d} / {cls_total:4d} ({cls_errors/cls_total*100:5.2f}% error rate)")

In [ ]:
# Show 15 misclassified examples for categorization
sample_errors = errors.sample(min(15, len(errors)), random_state=SEED)

print("\n" + "="*80)
print("MISCLASSIFIED EXAMPLES (for error pattern analysis)")
print("="*80)

for idx, (i, row) in enumerate(sample_errors.iterrows(), 1):
    true_label = {0: 'Hate', 1: 'Offensive', 2: 'Neither'}[row['class']]
    pred_label = {0: 'Hate', 1: 'Offensive', 2: 'Neither'}[row['predicted']]
    
    print(f"\nExample {idx}:")
    print(f"  Text: {row['text_clean']}")
    print(f"  True: {true_label} | Predicted: {pred_label} | Confidence: {row['confidence']:.3f}")
    print(f"  Category: [TO FILL - sarcasm/slang/context/ambiguous/annotation error?]")

## 15. Confidence Analysis

In [ ]:
# Confidence distribution
correct = results_df[results_df['correct']]
incorrect = results_df[~results_df['correct']]

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(correct['confidence'], bins=30, alpha=0.7, label='Correct', color='green', edgecolor='black')
plt.hist(incorrect['confidence'], bins=30, alpha=0.7, label='Incorrect', color='red', edgecolor='black')
plt.xlabel('Confidence', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Confidence Distribution', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

# Accuracy vs Coverage (like in Week 10)
thresholds = np.arange(0.5, 1.01, 0.05)
accuracies = []
coverages = []

for threshold in thresholds:
    above_threshold = results_df[results_df['confidence'] >= threshold]
    if len(above_threshold) > 0:
        acc = (above_threshold['correct'].sum() / len(above_threshold))
        cov = len(above_threshold) / len(results_df)
        accuracies.append(acc)
        coverages.append(cov)
    else:
        accuracies.append(0)
        coverages.append(0)

plt.subplot(1, 2, 2)
plt.plot(thresholds, accuracies, 'b-o', label='Accuracy', linewidth=2)
plt.plot(thresholds, coverages, 'r-o', label='Coverage', linewidth=2)
plt.xlabel('Confidence Threshold', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Accuracy vs Coverage Trade-off', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/confidence_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 16. Production Workflow Recommendations

Based on the confidence analysis above, fill in your recommendations:

### Confidence Thresholds
- **Auto-removal threshold** (hate speech): [e.g., 0.85 - high confidence]
- **Auto-approval threshold** (neither): [e.g., 0.75 - moderate confidence]
- **Human review range**: [e.g., confidence between 0.5 and threshold]

### Expected Workload (at 100K posts/day)
- **Auto-moderated**: [X%] = [Y] posts/day
- **Human review needed**: [Z%] = [W] posts/day

### Cost-Benefit Analysis
**False negatives (missing hate speech) are MORE costly because:**
- [Your reasoning]

**Therefore, we should:**
- [Lower threshold for hate speech detection? Accept more false positives?]

### Known Limitations
1. **Sarcasm/irony**: [How often does this cause failures?]
2. **Context-dependent language**: [Examples?]
3. **Evolving slang**: [How to handle?]
4. **Annotation disagreement**: [Evidence from errors?]

### V2 Recommendations
1. [e.g., Collect more hate speech examples to improve minority class]
2. [e.g., Add multi-lingual support]
3. [e.g., Incorporate user history/context]
4. [e.g., Active learning on borderline cases]

## 17. Save Final Results

In [ ]:
# Save transformer results
transformer_results = {
    'model': 'DistilBERT',
    'test_accuracy': float(test_acc),
    'test_f1_macro': float(test_f1_macro),
    'test_f1_weighted': float(test_f1_weighted),
    'confusion_matrix': cm.tolist(),
    'training_history': history
}

with open('../data/transformer_results.json', 'w') as f:
    json.dump(transformer_results, f, indent=2)

print("✅ Results saved to ../data/transformer_results.json")
print("✅ Model saved to ../data/best_model.pt")
print("\n🎉 Mini Project 9 - Transformer fine-tuning COMPLETE!")

---

## Summary

**DistilBERT Results:**
- Test Accuracy: [fill in]
- F1 (macro): [fill in]
- F1 (weighted): [fill in]

**vs TF-IDF Baseline:**
- Improvement: [+X.XX% accuracy]
- Best at: [which class?]
- Still struggles with: [which class? Why?]

**Next:** Write report and README for submission!